<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/Whisper_Notebooks/blob/main/Whisper_Bulk_Transcriber_AltTimestampMethod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Bulk Transcriber — Timestamped Transcript Variant (Optimized)

Transcribe audio and video files into **millisecond-accurate, timestamped** plain text using OpenAI's Whisper `large-v2` model.

This variant differs from the standard transcriber in one key way: the `.txt` output includes **DTW-aligned timestamps** on every line — sourced directly from when words are spoken in the audio, not rounded token predictions.

---

## What makes this different

| | Standard `Whisper_Bulk_Transcriber` | This notebook |
|---|---|---|
| `.txt` content | Plain text, no timestamps | `[HH:MM:SS.mmm --> HH:MM:SS.mmm]  spoken text` |
| Timestamp source | Model token predictions (20 ms steps) | Dynamic Time Warping on cross-attention weights |
| How accurate | ±20–60 ms | Word-boundary aligned to actual audio |
| `.srt` / `.vtt` | Included | Included (also DTW-refined) |

---

## Performance Tuning Flags

At the top of the code cell you can toggle five flags:

| Flag | Default | What it does | Typical gain |
|---|---|---|---|
| `PRECONVERT_AUDIO` | `True` | Pre-converts every file to 16 kHz mono WAV before transcription, eliminating Whisper's per-chunk resampling overhead. Especially helpful for MP3/AAC/M4A inputs. | 10–30% |
| `CUDNN_BENCHMARK` | `True` | Runs cuDNN's autotuner once to select the fastest convolution kernels for your audio chunk size. Pays off on batches. | small |
| `USE_INFERENCE_MODE` | `True` | Wraps transcription in `torch.inference_mode()` to skip autograd bookkeeping. Free speedup with no quality impact. | 3–10% |
| `FAST_DECODE` | `False` | Switches to greedy decoding (beam\_size=1, temperature=0) with fallback retries disabled. Much faster on clean audio; not recommended for noisy recordings. | up to 2× |
| `TORCH_COMPILE` | `False` | Experimental: `torch.compile(model)` via TorchDynamo/Inductor. Adds ~60s warmup on first call. Gains are inconsistent in Colab. | 5–15% |

### Expected VRAM usage on T4

| Mode | VRAM |
|---|---|
| FP16 (fp16=True, CUDA) | ~3–4 GB model + ~1–2 GB activations |
| FP32 (fp16=False) | ~6–8 GB — **avoid on T4** |

> `fp16` is set automatically based on whether CUDA is available.

---

## How To Use (3 Steps)

### Step 1 — Enable a free GPU
**Runtime → Change runtime type → Hardware accelerator → T4 GPU**

### Step 2 — Run the cell for the first time
Click **▶** on the code cell below. It will create the upload folder and stop with:
> *"Directory created. Upload your audio/video files there, then run this cell again."*

### Step 3 — Upload files and run again
1. Open the **Files panel** (📁 in the left sidebar)
2. Navigate to `/content/bulk_process_audios_here`
3. **Drag and drop** your audio or video files in
4. Click **▶** again — transcription starts automatically

> **Re-running the cell** is safe — if the model is already loaded, it will be reused without reloading.

---

## Output

For each source file, three outputs are saved to `/content/audio_transcription/`:

| Format | Content |
|---|---|
| `.txt` | `[00:00:00.000 --> 00:00:04.820]  Welcome everyone, today we're going to...` |
| `.srt` | Subtitle format with DTW-refined timestamps |
| `.vtt` | Web caption format with DTW-refined timestamps |

All files are zipped to `/content/all_transcribed_files.zip` — **download before closing the session**.

---

## Supported formats
| Audio | Video |
|-------|-------|
| `.mp3` `.wav` `.m4a` `.flac` | `.mp4` `.mov` `.avi` `.mkv` |
| `.aac` `.ogg` `.wma` `.opus` | `.webm` |
| `.aiff` `.amr` `.au` | |

> 🎬 Video files are handled automatically — the audio track is extracted and resampled before transcription.

---

## Troubleshooting

| Issue | Fix |
|---|---|
| Transcription very slow | Switch to GPU: **Runtime → Change runtime type → T4 GPU** |
| `ffmpeg` not found | Add a cell above: `!apt install -y ffmpeg` |
| `CUDA out of memory` on re-run | Model is already loaded — this is now handled automatically. If it persists, do **Runtime → Restart session** and re-run. |
| GPU out of memory (first run) | **Runtime → Restart session**, then re-run |
| `torch.compile` error | Set `TORCH_COMPILE = False` at the top of the cell |
| File skipped `[SKIP]` | File is empty (0 bytes) or unreadable — re-upload it |
| Session expired before download | Re-run the cell — all files are re-transcribed and re-zipped |

In [ ]:
# @title ## Whisper Bulk Transcriber — Timestamped (Optimized)
# @markdown ### Outputs `.txt` with DTW-aligned millisecond timestamps, plus `.srt` and `.vtt`.
# @markdown Final zip saved to `/content/all_transcribed_files.zip`.
# @markdown **Re-running this cell is safe** — the model is reused if already loaded.

# ── Phase 0: Constants & Performance Tuning Flags ────────────────────────────
# Only stdlib modules needed here — no pip installs required for directory check.

import os
import sys

BULK_DIR   = "/content/bulk_process_audios_here"
OUTPUT_DIR = "/content/audio_transcription"
ZIP_PATH   = "/content/all_transcribed_files.zip"
USE_MODEL  = "large-v2"

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

SUPPORTED_EXTENSIONS = {
    '.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.wma',
    '.aiff', '.opus', '.amr', '.au',
} | VIDEO_EXTENSIONS

TRANSCRIBED_EXTS = {'.txt', '.srt', '.vtt'}

# ── Performance Tuning ────────────────────────────────────────────────────────
# Adjust these flags before running to trade speed against quality.

# Pre-convert every audio file to 16 kHz mono WAV before passing it to Whisper.
# Eliminates Whisper's internal ffmpeg resampling overhead on every 30-second
# chunk. Pays off most on compressed formats (MP3, AAC, M4A) because those
# formats are CPU-bound to decode. WAV inputs already at 16 kHz are passed
# through unchanged. Recommended: True.
PRECONVERT_AUDIO   = True  # @param {type:"boolean"}

# Enable cuDNN autotuner. On first run, PyTorch benchmarks several kernel
# implementations and keeps the fastest. Helps on batches of similarly-sized
# audio chunks (e.g., 30-second segments). Negligible overhead otherwise.
CUDNN_BENCHMARK    = True  # @param {type:"boolean"}

# Wrap transcription in torch.inference_mode() to disable autograd bookkeeping.
# Free ~3-10% speedup with zero quality impact. Always safe to leave on.
USE_INFERENCE_MODE = True  # @param {type:"boolean"}

# Fast decode mode: greedy decoding (temperature=0, beam_size=1), fallback
# retries disabled. Up to 2x faster on clean speech. Not recommended for
# noisy, heavily accented, or poor-quality audio where fallback retries
# meaningfully improve accuracy.
FAST_DECODE        = False  # @param {type:"boolean"}

# Experimental: torch.compile(model). Can give 5-15% on T4 for
# decoder-heavy workloads. Adds ~60s warmup on first call. Can be unstable
# in Colab — disable if you see cryptic errors after model load.
TORCH_COMPILE      = False  # @param {type:"boolean"}

# ── Phase 1: Directory Check (runs FIRST — before any installs) ───────────────
# Instant and dependency-free. Saves the user from waiting through pip installs
# and model loading when there is nothing to do yet.

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(BULK_DIR):
    os.makedirs(BULK_DIR)
    sys.exit(
        f"\nDirectory created: '{BULK_DIR}'\n"
        f"Upload your audio/video files there, then run this cell again.\n"
    )

_all_in_dir = [
    os.path.join(BULK_DIR, f) for f in os.listdir(BULK_DIR)
    if os.path.isfile(os.path.join(BULK_DIR, f))
]
audio_files = sorted([
    f for f in _all_in_dir
    if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
])

if not audio_files:
    sys.exit(
        f"\nNo supported audio/video files found in '{BULK_DIR}'.\n"
        f"Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
        f"Upload files there and run this cell again.\n"
    )

print(f"Found {len(audio_files)} file(s) to transcribe. Loading dependencies...")

# ── Phase 2: Full Imports ─────────────────────────────────────────────────────
# Only reached when files are confirmed present.

import contextlib
import subprocess
import tempfile
import time
import unicodedata
import zipfile

# ── Phase 3: Library Check / Install ─────────────────────────────────────────
# Only openai-whisper is needed; torch/numpy/tqdm are installed as transitive
# dependencies. pydub, ffmpeg-python, and openai are NOT used by this notebook.
#
# PyTorch 2.x is recommended for SDPA (scaled dot-product attention) and
# fused kernels. Colab ships a compatible version, so no explicit torch install
# is needed. openai-whisper will pull the correct torch version automatically.

def _install_libraries():
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--root-user-action=ignore",
        "openai-whisper",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr)
        sys.exit(1)

try:
    import whisper
    import torch
    print("Libraries already installed. Skipping installation.")
except ImportError:
    print("Installing required libraries...")
    _install_libraries()
    try:
        import whisper
        import torch
        print("Libraries installed and imported successfully.")
    except ImportError as e:
        print(f"[ERROR] Import failed after installation: {e}")
        sys.exit(1)

# ── Phase 4: ffmpeg Binary Pre-flight Check ───────────────────────────────────

_ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True
)
if _ffmpeg_check.returncode != 0:
    print("[ERROR] ffmpeg binary not found on PATH.")
    print("Fix: add a cell above with   !apt install -y ffmpeg   and run it first.")
    sys.exit(1)
print("ffmpeg binary confirmed.")

# ── Phase 4b: Environment Diagnostics ────────────────────────────────────────
# Print device and PyTorch version so the user can verify the runtime is set up
# correctly. PyTorch 2.x enables SDPA (scaled dot-product attention) and fused
# convolution kernels that moderately improve T4 throughput.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device         : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"PyTorch        : {torch.__version__}")

# ── Phase 5: Per-file Pre-validation ─────────────────────────────────────────

valid_audio_files = []
for _f in audio_files:
    _name = os.path.basename(_f)
    if not os.access(_f, os.R_OK):
        print(f"[SKIP] Not readable (check permissions): {_name}")
        continue
    if os.path.getsize(_f) == 0:
        print(f"[SKIP] Empty file (0 bytes): {_name}")
        continue
    valid_audio_files.append(_f)

if not valid_audio_files:
    sys.exit("[ERROR] No valid audio files remain after validation. Aborting.")

audio_files = valid_audio_files
print(f"{len(audio_files)} file(s) passed validation.")

# ── Phase 6: Model Load ───────────────────────────────────────────────────────

from whisper.utils import format_timestamp, get_writer  # noqa: E402

if DEVICE == "cpu":
    _total_secs = 0.0
    _duration_known = True
    for _af in audio_files:
        _probe = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                _af,
            ],
            capture_output=True, text=True,
        )
        try:
            _total_secs += float(_probe.stdout.strip())
        except ValueError:
            _duration_known = False
            break

    _EST_SLOWDOWN = 20
    _border = "!" * 64
    print(f"\n{_border}")
    print("  WARNING: NO GPU DETECTED — TRANSCRIPTION WILL BE")
    print("  OVERWHELMINGLY SLOW (roughly 1 transcribed line per minute).")
    print()
    if _duration_known and _total_secs > 0:
        _audio_mins = _total_secs / 60
        _est_mins   = (_total_secs * _EST_SLOWDOWN) / 60
        _est_hrs    = _est_mins / 60
        print(f"  Total audio duration : ~{_audio_mins:.0f} min")
        if _est_hrs >= 1:
            print(f"  Estimated CPU time   : ~{_est_hrs:.1f} hours  (could be longer)")
        else:
            print(f"  Estimated CPU time   : ~{_est_mins:.0f} minutes  (could be longer)")
        print()
    print("  STRONGLY RECOMMENDED: switch to a GPU runtime first.")
    print("  Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print(f"{_border}\n")

    _answer = input(
        "Continue on CPU anyway? This will take an extremely long time. [y/N]: "
    ).strip().lower()

    if _answer != "y":
        sys.exit(
            "\nAborted. To enable GPU:\n"
            "  Runtime > Change runtime type > Hardware accelerator > T4 GPU\n"
            "Then run this cell again.\n"
        )
    print("Acknowledged. Continuing on CPU — this will take a very long time.\n")

# Guard: skip load if model already resident on the correct device.
# Re-running the cell retains `model` in the kernel namespace. Calling
# whisper.load_model() again double-allocates ~3 GB of VRAM before the
# old reference is GC'd, causing OOM or CUDA context corruption.
_need_load = True
if "model" in globals() and hasattr(model, "transcribe"):
    try:
        _loaded_device = str(next(model.parameters()).device).split(":")[0]
        if _loaded_device == DEVICE:
            print(f"Model '{USE_MODEL}' already loaded on {DEVICE}. Reusing.")
            _need_load = False
        else:
            print(
                f"Model is on '{_loaded_device}' but DEVICE is '{DEVICE}'. "
                "Releasing and reloading..."
            )
            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    except Exception:
        print("Could not verify existing model state. Reloading to be safe.")
        try:
            del model
        except NameError:
            pass

if _need_load:
    print(f"Loading '{USE_MODEL}' model on {DEVICE}...")
    try:
        model = whisper.load_model(USE_MODEL, device=DEVICE)
        print(f"Model '{USE_MODEL}' loaded on {DEVICE}.")
    except Exception as e:
        print(f"[ERROR] Failed to load Whisper model: {e}")
        sys.exit(1)

# cuDNN autotuner — benchmark kernel implementations once, reuse fastest.
# Effective when many similarly-sized audio chunks hit the mel-spectrogram
# convolutions. No downside except a tiny overhead on the very first chunk.
if CUDNN_BENCHMARK and DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print("cuDNN benchmark mode : enabled")

# torch.compile() — optional experimental compilation via TorchDynamo/Inductor.
# On T4 this can yield 5-15% for decoder-heavy workloads, but adds ~60s warmup
# and is sometimes unstable in Colab. Disabled by default.
if TORCH_COMPILE:
    try:
        print("Compiling model with torch.compile() — warmup may take ~60s on first call...")
        model = torch.compile(model)
        print("Model compiled successfully.")
    except Exception as e:
        print(f"[WARN] torch.compile() failed (non-fatal, continuing without): {e}")

if DEVICE == "cuda":
    _vram_used  = torch.cuda.memory_allocated() / 1e9
    _vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {_vram_used:.2f} GB used / {_vram_total:.2f} GB total")

# ── Phase 7: LiveFileWriter (console-only) ────────────────────────────────────
#
# Streams Whisper's verbose output to the console in real-time.
# The .txt file is written AFTER transcription completes using the
# DTW-aligned segment data from result['segments'] — not by parsing printed text.

class LiveFileWriter:
    "Tees sys.stdout to the console only. .txt is written after transcription."

    def __init__(self, stream):
        self.stream = stream

    def write(self, data):
        if data is None:
            return
        self.stream.write(data)
        self.stream.flush()

    def writelines(self, datas):
        for d in datas:
            self.write(d)

    def close(self):
        pass  # no file handle to close

    def __getattr__(self, name):
        return getattr(self.stream, name)


# ── Phase 7b: Audio Preparation Helper ───────────────────────────────────────
#
# Converts any audio/video file to a 16 kHz mono PCM WAV before transcription.
#
# Why 16 kHz mono WAV?
#   Whisper internally resamples every audio input to 16 kHz mono via ffmpeg.
#   Doing this once upfront (outside the transcribe loop) eliminates repeated
#   resampling overhead on each 30-second chunk. Especially impactful for
#   compressed formats (MP3, AAC, M4A) where decoding is CPU-bound and Colab
#   CPUs are slow relative to the T4 GPU.
#
# Optimisation path:
#   video file  → strip video stream + resample to 16 kHz mono WAV
#   audio file  → resample to 16 kHz mono WAV  (if PRECONVERT_AUDIO=True)
#   .wav file already at 16 kHz mono → returned as-is (no conversion)

def _is_already_optimal_wav(path: str) -> bool:
    "Return True if the file is already a 16 kHz mono PCM WAV (no conversion needed)."
    if os.path.splitext(path)[1].lower() != ".wav":
        return False
    _probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "a:0",
            "-show_entries", "stream=codec_name,sample_rate,channels",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True, text=True,
    )
    info = _probe.stdout.strip().split("\n")
    return (
        len(info) >= 3
        and info[0].strip() == "pcm_s16le"
        and info[1].strip() == "16000"
        and info[2].strip() == "1"
    )


def _prepare_audio(source_path: str, is_video: bool) -> tuple:
    # Convert source_path to a 16 kHz mono WAV suitable for Whisper.
    # Returns (tmp_path, converted) where:
    #   tmp_path  - path to the WAV file to pass to model.transcribe()
    #   converted - True if a temp file was created (must be deleted after use)
    # If the source is already an optimal WAV, returns (source_path, False).
    if not is_video and _is_already_optimal_wav(source_path):
        return source_path, False

    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        prefix="_whisper_pcm_",
        delete=False,
    )
    tmp_path = tmp.name
    tmp.close()

    cmd = [
        "ffmpeg", "-y",
        "-i", source_path,
        "-vn",                   # discard video stream (no-op for pure audio)
        "-acodec", "pcm_s16le",  # 16-bit signed PCM — Whisper's native format
        "-ar", "16000",          # 16 kHz sample rate
        "-ac", "1",              # mono
        tmp_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        err = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else "unknown ffmpeg error"
        raise RuntimeError(f"Audio preparation failed: {err}")

    return tmp_path, True


# ── Phase 8: Timestamped .txt Writer ─────────────────────────────────────────
#
# Writes the .txt file from result['segments'] AFTER transcription completes.
#
# Why this is more accurate than parsing verbose console text:
#   word_timestamps=True causes Whisper's timing.py to run Dynamic Time Warping
#   on the decoder's cross-attention weights, then overwrite:
#       segment["start"] = words[0]["start"]
#       segment["end"]   = words[-1]["end"]
#   These float values are word-boundary-aligned to the actual audio — not the
#   decoder's timestamp token predictions (which have fixed 20 ms resolution).

def _write_timestamped_txt(segments: list, output_path: str) -> None:
    "Write a timestamped plain-text transcript from DTW-aligned Whisper segments."
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp(seg["start"], always_include_hours=True)
            end   = format_timestamp(seg["end"],   always_include_hours=True)
            text  = seg["text"].strip()
            if text:
                f.write(f"[{start} --> {end}]  {text}\n")


# ── Phase 9: Transcription Options ───────────────────────────────────────────
#
# Two presets controlled by FAST_DECODE:
#
#   FAST_DECODE=False (default — quality mode):
#     beam_size=5 activates beam search. best_of=5 is intentionally kept for
#     compatibility but is ignored when beam_size > 1 (Whisper uses beam search
#     and disables stochastic sampling). temperature is a 6-step fallback retry
#     schedule — Whisper retries a segment with the next temperature value when
#     the output fails compression-ratio or logprob quality checks.
#
#   FAST_DECODE=True (speed mode):
#     Greedy decoding (beam_size=1, temperature=0.0). Fallback retries disabled
#     by setting all thresholds to None. Up to 2x faster on clean speech.
#     Not recommended for noisy or low-quality audio.

if FAST_DECODE:
    _options = {
        "task":                       "transcribe",
        "fp16":                       DEVICE == "cuda",
        # Greedy decoding — fastest possible decode path.
        # beam_size=1 disables beam search; temperature=0 disables sampling.
        "beam_size":                  1,
        "temperature":                0.0,
        # Disable fallback retries — prevents repeated decode passes on each
        # segment. Significant speedup on noisy audio that would otherwise
        # trigger multiple retries.
        "compression_ratio_threshold": None,
        "logprob_threshold":          None,
        "no_speech_threshold":        0.6,
        "condition_on_previous_text": True,
        "initial_prompt":             None,
        #"language":                   None,
        "language": "ja",
        # DTW word-level alignment — kept even in fast mode because it does not
        # add a separate decode pass; it runs hooks during the existing forward
        # pass with minimal overhead.
        "word_timestamps":            True,
    }
    print("FAST_DECODE mode: greedy decoding, fallback retries disabled.")
else:
    _options = {
        "task":                       "transcribe",
        "fp16":                       DEVICE == "cuda",
        # beam_size=5 activates beam search (5 beams). This is the primary
        # quality lever — set to 1 for greedy decoding (~5x faster decode).
        "beam_size":                  5,
        # best_of is only active when beam_size is None or 1 (stochastic
        # sampling). With beam_size=5, best_of is silently ignored by Whisper's
        # decoder. Kept here for future compatibility if beam_size is changed.
        "best_of":                    5,
        # temperature is a fallback retry schedule: if a segment fails quality
        # checks (compression ratio, logprob), Whisper retries with the next
        # temperature. 0.0 = greedy first attempt, higher = more random.
        "temperature":                (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        "condition_on_previous_text": True,
        "initial_prompt":             None,
        "language":                   None,
        "no_speech_threshold":        0.4,
        "logprob_threshold":          -1.5,
        "compression_ratio_threshold": 2.2,
        # Enables DTW cross-attention alignment (timing.py).
        # Overwrites seg['start']/seg['end'] with word-boundary-aligned
        # timestamps derived from which audio frames the model attended to
        # per word. Memory overhead is per-segment (hooks removed after each).
        "word_timestamps":            True,
    }

# torch.inference_mode() context — disables autograd bookkeeping for a free
# ~3-10% speedup. contextlib.nullcontext() is used when the flag is off so
# the transcription call site is identical either way.
_infer_ctx = torch.inference_mode() if USE_INFERENCE_MODE else contextlib.nullcontext()
if USE_INFERENCE_MODE:
    print("inference_mode : enabled")

# Print active settings summary
_mode_label = "FAST (greedy)" if FAST_DECODE else "QUALITY (beam search)"
print(f"Decode mode    : {_mode_label}")
print(f"Audio preconv  : {'enabled' if PRECONVERT_AUDIO else 'disabled'}")

# ── Phase 10: Transcription Loop ─────────────────────────────────────────────

succeeded_paths = set()   # tracks files transcribed successfully this run
failed_files    = {}
original_stdout = sys.stdout
n = len(audio_files)

for i, audio_path in enumerate(audio_files, 1):
    raw_stem    = os.path.splitext(os.path.basename(audio_path))[0]
    output_stem = unicodedata.normalize("NFC", raw_stem)
    txt_path    = os.path.join(OUTPUT_DIR, output_stem + ".txt")

    print(f"\n[{i}/{n}] Processing: {os.path.basename(audio_path)}")

    # --- Audio preparation ---------------------------------------------------
    # For video files: strip video stream + resample to 16 kHz mono WAV.
    # For audio files: resample to 16 kHz mono WAV (if PRECONVERT_AUDIO=True).
    # Files already in optimal WAV format are detected and passed through as-is.
    # Keeping the model loaded ("hot") between files avoids CUDA graph rebuilds
    # and is the most impactful single architectural optimisation.
    _temp_audio = None
    transcribe_path = audio_path
    _is_video = os.path.splitext(audio_path)[1].lower() in VIDEO_EXTENSIONS

    if _is_video or PRECONVERT_AUDIO:
        _label = "Video — extracting audio" if _is_video else "Pre-converting to 16 kHz mono WAV"
        print(f"  {_label}...")
        try:
            _prepared, _converted = _prepare_audio(audio_path, is_video=_is_video)
            if _converted:
                _temp_audio     = _prepared
                transcribe_path = _prepared
                print(f"  Audio ready.")
            else:
                print(f"  Already 16 kHz mono WAV — skipping conversion.")
        except RuntimeError as e:
            msg = str(e)
            print(f"[FAIL] {os.path.basename(audio_path)}: {msg}")
            failed_files[audio_path] = msg
            continue

    # .txt is NOT pre-cleared — if transcription fails, the previous successful
    # .txt is preserved. _write_timestamped_txt truncates on successful write.

    live_writer = LiveFileWriter(original_stdout)
    result = None
    _t0 = time.time()

    try:
        sys.stdout = live_writer
        with _infer_ctx:
            result = model.transcribe(transcribe_path, verbose=True, **_options)

    except RuntimeError as e:
        msg = str(e)
        if "out of memory" in msg.lower():
            torch.cuda.empty_cache()
            msg = (
                "GPU out of memory. "
                "Try a shorter audio file, or restart the runtime to free VRAM: "
                "Runtime > Restart session."
            )
        else:
            msg = f"RuntimeError: {e}"
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    except Exception as e:
        msg = str(e)
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    finally:
        sys.stdout = original_stdout
        live_writer.close()
        if _temp_audio and os.path.exists(_temp_audio):
            os.unlink(_temp_audio)

    if result is None:
        continue

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    if not result.get("segments"):
        print(f"  No speech detected in: {os.path.basename(audio_path)}")

    succeeded_paths.add(audio_path)

    # Write .txt from DTW-aligned segment data.
    if result.get("segments"):
        _write_timestamped_txt(result["segments"], txt_path)
        print(f"  Saved: {output_stem}.txt")

    print(f"  Completed in {time.time() - _t0:.1f}s")

    # Write .srt and .vtt. These also benefit from DTW-refined boundaries
    # because get_writer reads result['segments'] which word_timestamps=True
    # has already overwritten with word-aligned start/end values.
    norm_audio_path = os.path.join(
        os.path.dirname(audio_path),
        output_stem + os.path.splitext(audio_path)[1]
    )

    for fmt in ("srt", "vtt"):
        try:
            fix_vtt = (
                fmt == "vtt"
                and bool(result.get("segments"))
                and result["segments"][0].get("start") == 0
            )
            # Save the original start value so we can restore it precisely
            # after writing — avoids corrupting the segment data in memory.
            _orig_start = result["segments"][0]["start"] if fix_vtt else None
            if fix_vtt:
                result["segments"][0]["start"] += 1 / 1000

            try:
                writer = get_writer(fmt, OUTPUT_DIR)
                writer(result, norm_audio_path)
            finally:
                if fix_vtt:
                    result["segments"][0]["start"] = _orig_start

            print(f"  Saved: {output_stem}.{fmt}")

        except Exception as e:
            print(f"  [WARN] Could not write .{fmt} for '{output_stem}': {e}")

# ── Phase 11: Zip All Transcribed Files ──────────────────────────────────────

if not succeeded_paths:
    print("\n[ERROR] No files transcribed successfully. Nothing to zip.")
else:
    _files_to_zip = sorted([
        os.path.join(OUTPUT_DIR, fname)
        for fname in os.listdir(OUTPUT_DIR)
        if os.path.splitext(fname)[1].lower() in TRANSCRIBED_EXTS
    ])

    if not _files_to_zip:
        print("\n[WARN] Output directory has no .txt/.srt/.vtt files to zip.")
    else:
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for fpath in _files_to_zip:
                zf.write(fpath, arcname=os.path.basename(fpath))

        _zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
        print(
            f"\nZipped {len(_files_to_zip)} file(s) -> {ZIP_PATH} "
            f"({_zip_mb:.2f} MB)"
        )
        print(
            "REMINDER: Download your zip before the Colab session ends. "
            "/content/ is erased when the runtime disconnects."
        )

# ── Phase 12: Final Summary ───────────────────────────────────────────────────

print("\n" + "=" * 52)
print("TRANSCRIPTION SUMMARY")
print(f"  Total files : {n}")
print(f"  Succeeded   : {len(succeeded_paths)}")
print(f"  Failed      : {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for _path, _reason in failed_files.items():
        print(f"  x {os.path.basename(_path)}")
        print(f"    Reason: {_reason}")
print("=" * 52)


SystemExit: 
Directory created: '/content/bulk_process_audios_here'
Upload your audio/video files there, then run this cell again.


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
